# Unidad 4 — VA bidimensionales y suma de variables

Cuando trabajamos con **varias variables a la vez** aparecen dos preguntas centrales:

1. ¿Cómo se distribuye la **suma** $S_n = X_1 + \dots + X_n$?
2. ¿Qué pasa con la **media muestral** $\bar{X}_n$ cuando $n$ crece?

Las dos respuestas son **el corazón estadístico de la materia**:

- **LLN (Ley de los grandes números):** $\bar{X}_n \to \mu$ casi seguramente.
- **TCL (Teorema central del límite):** $\sqrt{n}\,(\bar{X}_n - \mu) \to \mathcal{N}(0, \sigma^2)$.


## Importaciones

In [ ]:
from core import BinomialParams, ExponentialParams, Settings
from distributions import make_binomial, make_exponential
from exercises import NumericAnswerInput, verify_numeric_answer
from sampling import (
    CLTSimulationInput,
    GaltonBoardInput,
    LLNSimulationInput,
    simulate_clt,
    simulate_galton_board,
    simulate_lln,
)
from visualization import (
    CLTComparisonChartInput,
    LLNChartInput,
    chart_clt_comparison,
    chart_lln_running_mean,
)
from widgets import (
    CLTExplorerInput,
    LLNExplorerInput,
    build_clt_explorer,
    build_lln_explorer,
)

In [ ]:
settings = Settings()

## Concreto: tirar una moneda y promediar

Sea $X_i$ una Bernoulli($p = 0{,}3$): cara=1, ceca=0. La media muestral $\bar{X}_n$ es
**la proporción observada de caras** después de $n$ tiradas.

Si $p$ es la "verdad", la LLN promete que $\bar{X}_n \to 0{,}3$ cuando $n \to \infty$.


In [ ]:
bernoulli = make_binomial(BinomialParams(trials=1, success_probability=0.3))
lln_result = simulate_lln(LLNSimulationInput(distribution=bernoulli, horizon=4_000, settings=settings))
chart_lln_running_mean(LLNChartInput(lln_result=lln_result, settings=settings))

### Intuición Singapur — "se estabiliza"

Los datos al principio rebotan mucho. **No** porque las primeras tiradas sean diferentes,
sino porque al promediar pocos datos cada nueva observación pesa mucho. Después de unos
miles, una tirada extra cambia la media en un orden de magnitud despreciable.


In [ ]:
build_lln_explorer(LLNExplorerInput(settings=settings))

## Concreto: tablero de Galton

El **tablero de Galton** es la metáfora pictórica del TCL. Cada bola toma 20 decisiones
independientes izquierda/derecha. La **posición final** es una **suma** de 20 ±1, y se
distribuye aproximadamente como una Normal — aunque cada paso individual es uniforme.


In [ ]:
import altair as alt
import pandas as pd

galton_result = simulate_galton_board(GaltonBoardInput(rows=24, balls=8_000, settings=settings))
galton_table = pd.DataFrame({
    "posición": galton_result.bin_positions,
    "frecuencia": galton_result.bin_counts,
})
alt.Chart(galton_table).mark_bar(opacity=0.75, color="#1f77b4").encode(
    x=alt.X("posición:O", title="Casilla"),
    y=alt.Y("frecuencia:Q", title="Bolas"),
).properties(width=520, height=320, title="Tablero de Galton — suma de 24 pasos ±1")

## Pictórico: el TCL en acción

Tomamos $n$ muestras de una **Exponencial** (que está muy lejos de ser Normal: tiene
cola larga y es asimétrica). Calculamos la media. **Repetimos miles de veces** y dibujamos
la distribución de las **medias estandarizadas**. La cosa converge a $\mathcal{N}(0, 1)$
incluso aunque la fuente sea "fea".


In [ ]:
exponential_distribution = make_exponential(ExponentialParams(rate=1.0))
clt_result = simulate_clt(
    CLTSimulationInput(
        distribution=exponential_distribution,
        sample_size_per_replicate=30,
        replicates=5_000,
        settings=settings,
    )
)
chart_clt_comparison(CLTComparisonChartInput(clt_result=clt_result, settings=settings))

### Intuición Singapur — "promediar limpia la asimetría"

Cada exponencial tiene una cola larga a la derecha. Pero al **promediar 30 exponenciales**,
las colas se cancelan parcialmente: las muestras altas y bajas se compensan. Cuanto más
grande es $n$, menos se nota el origen de las muestras.


In [ ]:
build_clt_explorer(CLTExplorerInput(settings=settings))

## Abstracto: qué dice exactamente el TCL

Sean $X_1, X_2, \dots$ i.i.d. con $E[X_i] = \mu$ y $\mathrm{Var}(X_i) = \sigma^2 < \infty$.
Entonces:

$$ Z_n = \frac{\bar{X}_n - \mu}{\sigma / \sqrt{n}} \xrightarrow{d} \mathcal{N}(0, 1). $$

**Notar.** El TCL **no** dice que $X_i$ tienda a una Normal. Dice que la **media
estandarizada** tiende a una Normal. Es la *estructura del promedio* la que se vuelve Normal.


## Aproximación Normal a Binomial

Como consecuencia: si $Y \sim \text{Bin}(n, p)$, entonces $Y \approx \mathcal{N}(np, np(1-p))$
para $n$ grande (regla práctica: $np \ge 10$ y $n(1-p) \ge 10$).


## Ejercicio 1 — Distribución de la media muestral

Si $X_i \sim \mathcal{N}(50, 100)$ (desvío 10) y tomamos $n = 25$ observaciones,
¿cuál es el desvío estándar de $\bar{X}_n$?

**Idea.** $\sigma_{\bar{X}} = \sigma / \sqrt{n} = 10 / \sqrt{25} = 2$.


In [ ]:
import math

expected_standard_error = 10.0 / math.sqrt(25)
student_answer = 2.0
verify_numeric_answer(NumericAnswerInput(student_answer=student_answer, expected_answer=expected_standard_error))

## Ejercicio 2 — Aproximación Binomial-Normal

$Y \sim \text{Bin}(n = 100, p = 0{,}4)$. Aproximar $P(Y \le 45)$ con la Normal
(sin corrección por continuidad para simplificar).

**Idea.** $E[Y] = 40$, $\mathrm{Var}(Y) = 24$. Estandarizamos:
$P(Y \le 45) \approx P\bigl(Z \le (45 - 40)/\sqrt{24}\bigr) \approx P(Z \le 1{,}02) \approx 0{,}846$.


In [ ]:
from core import NormalParams
from distributions import make_normal, tail_probability_of_continuous
from distributions.evaluations import TailProbabilityInput

normal_approximation = make_normal(NormalParams(mean=40.0, standard_deviation=math.sqrt(24.0)))
expected_probability = tail_probability_of_continuous(
    TailProbabilityInput(distribution=normal_approximation, upper_bound=45.0)
).probability

student_answer = 0.846
verify_numeric_answer(
    NumericAnswerInput(
        student_answer=student_answer,
        expected_answer=expected_probability,
        absolute_tolerance=5e-3,
    )
)

## Para llevarse

- LLN: el promedio se *clava* en $\mu$.
- TCL: el promedio **estandarizado** se vuelve Normal, sin importar la fuente.
- Esto justifica por qué la **Normal** aparece tan seguido en datos reales: muchos
  fenómenos son sumas/promedios de efectos chicos e independientes.
